In [ ]:
import pandas as pd
import numpy as np


# 1. Load the data
df = pd.read_csv('temporal_summary.csv')

# 2. Clean up column names by stripping hidden leading/trailing spaces
df.columns = df.columns.str.strip()

# Create a backup copy of the raw data
data = df.copy()

print("Data loaded! Here are the first 5 rows:")
display(df.head())

# Output the data types to ensure 'hour_of_day' is numeric
print("\nData Types:")
print(df.dtypes)


Patch applied. You can now run your binning code!
Data loaded! Here are the first 5 rows:


,aggregate_id,timestamp_hour,total_volume,month_of_year,day_of_week,hour_of_day,acuity_1,acuity_2,acuity_3,acuity_4,acuity_5,upload_id
0,1,2111-08-21 20:00:00,1,8,6,20,0,0,0,1,0,3
1,2,2112-09-30 01:00:00,1,9,6,1,0,1,0,0,0,3
2,3,2114-06-25 14:00:00,1,6,2,14,0,0,1,0,0,3
3,4,2115-05-08 18:00:00,1,5,4,18,0,0,1,0,0,3
4,5,2115-09-25 15:00:00,1,9,4,15,0,0,1,0,0,3



Data Types:
aggregate_id       int64
timestamp_hour    object
total_volume       int64
month_of_year      int64
day_of_week        int64
hour_of_day        int64
acuity_1           int64
acuity_2           int64
acuity_3           int64
acuity_4           int64
acuity_5           int64
upload_id          int64
dtype: object


In [18]:
# Force 'hour_of_day' to be numeric. 
# errors='coerce' turns any unparseable strings into NaN instead of crashing.
df['hour_of_day'] = pd.to_numeric(df['hour_of_day'], errors='coerce')

# Safely create the time_window column
df['time_window'] = df['hour_of_day'] // 4

# Group and aggregate as you originally did
df_binned = df.groupby(['upload_id', 'day_of_week', 'time_window'], as_index=False).agg({
    'total_volume': 'sum',
    'acuity_1': 'sum',
    'acuity_2': 'sum',
    'acuity_3': 'sum',
    'acuity_4': 'sum',
    'acuity_5': 'sum',
    'month_of_year': 'first'
})

print("Data Binned! Here is the new structure:")
display(df_binned.head(10))


AttributeError: module 'numpy' has no attribute 'matrix'

In [4]:
df.describe()


,aggregate_id,total_volume,month_of_year,day_of_week,hour_of_day,acuity_1,acuity_2,acuity_3,acuity_4,acuity_5,upload_id
count,658.000000,658.0,658.000000,658.000000,658.000000,658.000000,658.000000,658.000000,658.000000,658.0,658.000000
mean,329.500000,1.0,6.802432,3.841945,13.370821,0.039514,0.319149,0.554711,0.086626,0.0,5.504559
std,190.092521,0.0,3.533772,1.946609,5.688184,0.194962,0.466501,0.497376,0.281500,0.0,1.694510
min,1.000000,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3.000000
25%,165.250000,1.0,4.000000,2.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.0,4.000000
50%,329.500000,1.0,7.000000,4.000000,14.000000,0.000000,0.000000,1.000000,0.000000,0.0,6.000000
75%,493.750000,1.0,10.000000,5.000000,18.000000,0.000000,1.000000,1.000000,0.000000,0.0,7.000000
max,658.000000,1.0,12.000000,7.000000,23.000000,1.000000,1.000000,1.000000,1.000000,0.0,8.000000


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 658 entries, 0 to 657
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   aggregate_id    658 non-null    int64 
 1   timestamp_hour  658 non-null    object
 2   total_volume    658 non-null    int64 
 3   month_of_year   658 non-null    int64 
 4   day_of_week     658 non-null    int64 
 5   hour_of_day     658 non-null    int64 
 6   acuity_1        658 non-null    int64 
 7   acuity_2        658 non-null    int64 
 8   acuity_3        658 non-null    int64 
 9   acuity_4        658 non-null    int64 
 10  acuity_5        658 non-null    int64 
 11  upload_id       658 non-null    int64 
dtypes: int64(11), object(1)
memory usage: 61.8+ KB


In [19]:
!pip install "numpy<2.0.0"


   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   ------------- -------------------------- 5.2/15.5 MB 29.0 MB/s eta 0:00:01
   ----------------------------- ---------- 11.5/15.5 MB 27.8 MB/s eta 0:00:01
   ---------------------------------------- 15.5/15.5 MB 25.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3


  You can safely remove it manually.
  You can safely remove it manually.


In [16]:
# 1. We create a 'Window' column. 
# This divides the 24 hours of the day into 6 blocks (0, 1, 2, 3, 4, 5).
# Hour 0-3 = Block 0, Hour 4-7 = Block 1, etc.
data['time_window'] = data['hour_of_day'] // 4

# 2. Now we group by Upload, Day, and the new Window to sum up the volumes.
# This turns the "1s" into a "Total for that block".
data_binned = data.groupby(['upload_id', 'day_of_week', 'time_window']).agg({
    'total_volume': 'sum',
    'acuity_1': 'sum',
    'acuity_2': 'sum',
    'acuity_3': 'sum',
    'acuity_4': 'sum',
    'acuity_5': 'sum',
    'month_of_year': 'first' # Keep the month the same
}).reset_index()

print("Data Binned! Here is the new structure:")
display(data_binned.head(10))

AttributeError: module 'numpy' has no attribute 'matrix'